In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBRegressor
from SamplingMethods import Sampler_class

In [2]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelXG/ModelXG.json")

In [3]:
def SurrogateModelOfReality(n_ci, n_it):
    y_pred = loaded_model.predict(np.array([[n_ci],[n_it]]).T)[0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
class client(object):
    def __init__(self, df):
        self.df = df
    def summarize(self):
        return df

In [6]:
class RangeParameterConfig(object):
    def __init__(self, name, bounds):
        self.name = name
        self.bounds = bounds

In [7]:
class OptimisationSetup_class(object):
    def __init__(self):
        self.Parameters_lis = [
            RangeParameterConfig(name="s1", bounds=(0, 1)),
            RangeParameterConfig(name="s2", bounds=(0, 1)),
            RangeParameterConfig(name="b1", bounds=(0, 1)),
        ]
OptimisationSetup_obj = OptimisationSetup_class()

In [8]:
y_max_lis = []

for i in range(100):
    sampler_obj = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler_obj.three.QuasirandomSampler3D_func(8,Parameters_lis).T
    y = []
    for row in X:
        s1 = row[0]
        s2 = row[1]
        b1 = row[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        y.append(SurrogateModelOfReality(n_ci,n_it))
    y = np.array(y)
    d = {"s1": X.T[0], "s2": X.T[1], "b1": X.T[2], "t1": y}
    df = pd.DataFrame(data=d)
    client_obj = client(df)
    # client_obj.summarize()
    sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client_obj)

    for _ in range(19):
        trial = sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client_obj)
        s1 = trial[0][0]
        s2 = trial[0][1]
        b1 = trial[0][2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        result = SurrogateModelOfReality(n_ci,n_it)
        nd = {"s1": [s1], "s2": [s2], "b1": [b1], "t1": [result]}
        df_new_rows = pd.DataFrame(data=nd)
        df = pd.concat([df,df_new_rows],ignore_index=True)
        client_obj = client(df)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client_obj.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

Trial 0 =========================================
5.379894733428955

Trial 1 =========================================
2.3438973426818848

Trial 2 =========================================
1.6304852962493896

Trial 3 =========================================
4.402780055999756

Trial 4 =========================================
2.818509578704834

Trial 5 =========================================
3.7287399768829346

Trial 6 =========================================
1.7610442638397217

Trial 7 =========================================
4.206354141235352

Trial 8 =========================================
2.089252471923828

Trial 9 =========================================
1.6334861516952515

Trial 10 =========================================
3.584895610809326

Trial 11 =========================================
1.692118525505066

Trial 12 =========================================
2.9904398918151855

Trial 13 =========================================
4.164059638977051

Trial 14 ===============

In [9]:
y_max_arr = np.array(y_max_lis)
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 5.9023942947387695
Avg = 2.801213515996933
Std = 0.9633557468961327


In [10]:
print(y_max_arr.tolist())

[5.379894733428955, 2.3438973426818848, 1.6304852962493896, 4.402780055999756, 2.818509578704834, 3.7287399768829346, 1.7610442638397217, 4.206354141235352, 2.089252471923828, 1.6334861516952515, 3.584895610809326, 1.692118525505066, 2.9904398918151855, 4.164059638977051, 2.818509578704834, 2.4738709926605225, 4.517553806304932, 1.7735464572906494, 2.818509578704834, 2.3438973426818848, 4.104054927825928, 1.7062249183654785, 2.961571216583252, 2.894249439239502, 2.3438973426818848, 3.7358412742614746, 5.379894733428955, 2.818509578704834, 2.089252471923828, 2.7023568153381348, 2.7023568153381348, 2.5983049869537354, 2.818509578704834, 2.961571216583252, 1.6304852962493896, 2.71921443939209, 2.818509578704834, 1.9425209760665894, 2.7023568153381348, 3.7287399768829346, 2.1384005546569824, 1.7062249183654785, 3.130575656890869, 3.3295278549194336, 2.3438973426818848, 2.3438973426818848, 4.069692611694336, 2.9904398918151855, 1.6304852962493896, 2.044818878173828, 3.3295278549194336, 1.70

In [11]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelXG/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [12]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelXG/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

            0
0    3.329528
1    3.987710
2    3.584896
3    1.860339
4    3.329528
..        ...
295  2.769724
296  2.702357
297  5.902394
298  4.346411
299  3.008750

[300 rows x 1 columns]


In [13]:
# # Sanity check to make sure the MIPT is running correctly.
# df = client.summarize()
# types_lis = []
# for i in range(len(df)):
#     if i < 8:
#         types_lis.append("one-shot")
#     else:
#         types_lis.append("sequential")
# df["type"] = types_lis
# fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='type',width=1300, height=600)
# fig.show()